# Move Signpost-D checkpoints from Kaggle to Colab

Notebook 11's runs, notebook 14's evaluations and the benchmark touch counter all live in `checkpoints/` in the working area. This notebook moves that folder through a **private** Hugging Face repo.

- **On Kaggle (`WHERE = "kaggle"`)**, after notebook 11 has stopped: uploads `checkpoints/` to `REPO` (created private if it does not exist). It lists every run and its size first.
- **On Colab (`WHERE = "colab"`)**, before notebooks 11 or 14: mounts Google Drive, downloads `REPO` into `DRIVE_DIR` (skipped if already there), and links it as `checkpoints/` in `/content`. Every later checkpoint is then written to Drive, so a Colab disconnect resumes from Drive.

The token needs **write** access for the upload. It comes from the environment, Colab's secrets (`HF_TOKEN`) or a hidden prompt, and is never printed or saved.

In [3]:
# Parameters

WHERE = "kaggle"                               # "kaggle": upload checkpoints/ | "colab": download into Drive and link
REPO = "skamalj/signpost-d-checkpoints"   # a private model repo on Hugging Face
DRIVE_DIR = "/content/drive/MyDrive/signpost_d/checkpoints"   # Colab: where the checkpoints live from now on

In [4]:
# Token: environment, then Colab secrets, then a hidden prompt (never printed or saved)

import getpass, os, subprocess
from pathlib import Path

token, source = os.environ.get("HF_TOKEN"), "the environment"
if not token:
    try:
        from google.colab import userdata
        token, source = userdata.get("HF_TOKEN"), "Colab secret 'HF_TOKEN'"
    except Exception:
        pass
if not token:
    token, source = getpass.getpass("Hugging Face token with write access: ").strip(), "the prompt"
os.environ["HF_TOKEN"] = token
from huggingface_hub import HfApi, snapshot_download
api = HfApi(token=token)
print(f"Hugging Face token from {source}; signed in as {api.whoami()['name']}")
assert not REPO.startswith("YOUR_HF_USERNAME"), "set REPO to your own account first"

Hugging Face token from the prompt; signed in as skamalj


In [5]:
# Kaggle: list and upload checkpoints/

def du(p):
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())


if WHERE == "kaggle":
    ck = Path("checkpoints")
    assert ck.is_dir(), "run this in the Kaggle session whose working area holds notebook 11's checkpoints/"
    for d in sorted(p for p in ck.iterdir() if p.is_dir()):
        for run in sorted(p for p in d.iterdir() if p.is_dir()):
            state = "finished" if (run / "done.json").exists() else ("in progress" if list(run.glob("attempt-seed*/latest.pt")) else "?")
            print(f"  {d.name}/{run.name}: {du(run) / 1e6:.0f} MB, {state}")
    print(f"total {du(ck) / 1e9:.2f} GB; touch counter present: {(ck / 'test_touches.json').exists()}")
    api.create_repo(REPO, private=True, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=str(ck), repo_id=REPO, repo_type="model", ignore_patterns=["*.tmp"],
                      commit_message="notebook 11 / 14 checkpoints from Kaggle")
    print(f"uploaded to {REPO} (private)")

  all-pos-v1_3-lr0.0001-head0.00025-warm0.15-ep3/seed1042: 117 MB, finished
  all-pos-v1_3-lr0.0001-head0.00025-warm0.15-ep3/seed2042: 117 MB, finished
  all-pos-v1_3-lr0.0001-head0.00025-warm0.15-ep3/seed42: 117 MB, finished
  dec-Qwen3-0.6B-v1_3-zeroshot-pos1/zeroshot: 0 MB, finished
  dec-Qwen3-1.7B-v1_3-train-pos1/lr0.0001-r16-ep1-fp16-packed-seed42: 70 MB, finished
  dec-Qwen3-1.7B-v1_3-zeroshot-pos1/zeroshot: 0 MB, finished
  dec-Qwen3-4B-Instruct-2507-v1_3-soup-pos1/soup-v1_3-ep1-seeds42-1042-packed: 0 MB, finished
  dec-Qwen3-4B-Instruct-2507-v1_3-train-pos1/lr0.0001-r16-ep1-fp16-packed-seed1042: 132 MB, finished
  dec-Qwen3-4B-Instruct-2507-v1_3-train-pos1/lr0.0001-r16-ep1-fp16-packed-seed42: 132 MB, finished
  dec-Qwen3-4B-Instruct-2507-v1_6-train-pos1/lr0.0001-r16-ep2-fp16-packed-seed42: 132 MB, finished
  dec-Qwen3-4B-Instruct-2507-v1_6-train-pos1/lr0.0001-r64-ep2-fp16-packed-seed42: 529 MB, finished
  dec-Qwen3-4B-Instruct-2507-v1_6-train-pos1/lr0.0002-r16-ep2-fp16-packed-

In [6]:
# Colab: download into Drive once, then link it as /content/checkpoints

if WHERE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    dst = Path(DRIVE_DIR)
    if dst.exists() and any(dst.iterdir()):
        print(f"{dst} already has checkpoints: download skipped")
    else:
        dst.mkdir(parents=True, exist_ok=True)
        snapshot_download(REPO, repo_type="model", local_dir=str(dst), token=token)
        print(f"downloaded {REPO} into {dst}")
    link = Path("/content/checkpoints")
    if link.is_symlink() or link.exists():
        assert link.is_symlink() and link.resolve() == dst.resolve(), f"{link} exists and is not a link to {dst}: move it away first"
    else:
        link.symlink_to(dst, target_is_directory=True)
    runs = sorted(str(p.parent.relative_to(dst)) for p in dst.rglob("done.json") if "evaluate" not in str(p))
    print(f"{link} -> {dst}\nfinished runs ({len(runs)}):\n  " + "\n  ".join(runs))
    print(f"touch counter present: {(dst / 'test_touches.json').exists()}")